# Homework: Predicting High-Energy Seismic Bumps with Imbalanced Classification

## Scientific context

Each observation summarizes seismic and seismoacoustic activity during an 8-hour mining shift. The target indicates whether a high-energy seismic bump occurred during the following shift. Such events are rare, making this an **imbalanced binary-classification** problem.

The goal is to build and evaluate a transparent introductory classification workflow while recognizing that false negatives and false positives have different practical consequences.

## Learning objectives

By completing this assignment, you will be able to:

1. Define predictors and a binary target for a geoscience classification problem.
2. Create **stratified train, validation, and test sets** and explain their distinct purposes.
3. Prevent information leakage by fitting preprocessing steps only on training data.
4. Establish a naive baseline before fitting machine-learning models.
5. Compare an unweighted classifier with a class-weighted classifier.
6. Evaluate an imbalanced classifier using confusion matrices, recall, precision, F-scores, balanced accuracy, ROC-AUC, and average precision.
7. Select a probability threshold using validation data and evaluate the final choice once on the untouched test set.
8. Communicate model limitations and the consequences of classification errors.

## Submission

Submit this completed notebook with all cells run and outputs visible. Replace every **TODO** with your work. Keep the test set untouched until the final section.

## 0. Setup

The notebook uses `pandas`, `matplotlib`, and `scikit-learn`. Reading a Parquet file also requires `pyarrow` or `fastparquet`.

Install missing packages from a terminal or a separate notebook cell, for example:

```bash
pip install pandas pyarrow matplotlib scikit-learn
```

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
DATA_URL = "https://data.openml.org/datasets/0004/46956/dataset_46956.pq"
TARGET = "HighEnergySeismicBump"

## 1. Load and verify the dataset

In [ ]:
# Load directly from the Parquet URI.
df = pd.read_parquet(DATA_URL)
df.head()

In [ ]:
# TODO
# Basic integrity checks of column names, data types, missing values, and descriptive statistics (quantiles, min, max, mean, etc)

### Question 1 — Data meaning and prediction target

In 3–5 sentences:

- State the observational unit represented by one row.
- State what the model is trying to predict.
- Explain why this is a classification problem rather than a regression problem.
- Identify which outcome should be considered the positive class and why.

**Answer:**  
TODO

## 2. Inspect and encode the target

OpenML may store a nominal target as strings, booleans, integers, or a pandas categorical type. The following cell inspects the values without assuming their representation.

In [ ]:
df[TARGET].unique()

### Question 2 — Quantifying imbalance

Calculate the majority-to-minority class ratio. Then explain why a classifier that predicts the majority class for every observation could appear successful under ordinary accuracy.

Create a numeric target in which `1` means a high-energy seismic bump and `0` means no high-energy bump. The helper below handles common binary representations. **Inspect its output and confirm that the rare class is encoded as 1.**

In [ ]:
df["HighEnergySeismicBump"] = (
    df["HighEnergySeismicBump"]
    .map({"Yes": 1, "No": 0})
)


y = df[TARGET].astype(int)
X = df.drop(columns=TARGET)

# TODO: calculate the majority-to-minority ratio.

**Interpretation:**  
TODO

## 3. Create train, validation, and test sets

Use three disjoint subsets:

- **Training set:** fit preprocessing parameters and model coefficients.
- **Validation set:** compare models and select a probability threshold.
- **Test set:** estimate performance of the final, already-selected workflow.

We will allocate 60% to training, 20% to validation, and 20% to testing. Because the hazardous class is rare, use **stratification** in both splitting steps.

> This curated dataset is intended for an IID tabular classification study and does not supply a time field or site/group identifier. Therefore, this assignment uses stratified random splitting. In a real environmental or geological deployment, temporal, spatial, borehole, station, watershed, or site-level dependence may require blocked or grouped splitting instead.

In [ ]:
# First split: 80% development data, 20% final test data.
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Second split: divide development data into 75% train and 25% validation.
# This gives 60% train, 20% validation, and 20% test overall.
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_dev,
)

split_summary = pd.DataFrame({
    "rows": [len(y_train), len(y_val), len(y_test)],
    "positive_count": [y_train.sum(), y_val.sum(), y_test.sum()],
    "positive_fraction": [y_train.mean(), y_val.mean(), y_test.mean()],
}, index=["train", "validation", "test"])

display(split_summary)

### Question 3 — Splitting and leakage

Answer each prompt in 2–4 sentences.

1. Why is stratification useful in this dataset?
2. What distinct role does the validation set play?
3. Why must the test set not be used to choose a model, hyperparameter, class weight, or threshold?
4. Explain why scaling, imputation, feature selection, or resampling **before** splitting can leak information.
5. Give one realistic grouping or ordering variable that might require a non-random split in a future geoscience dataset.

**Answers:**

1. TODO
2. TODO
3. TODO
4. TODO
5. TODO

## 4. Build a leakage-safe preprocessing pipeline

Categorical predictors will be one-hot encoded. Numeric predictors will be median-imputed and standardized. Although this dataset has no missing values, including imputation makes the workflow more robust and demonstrates proper pipeline construction.

In [ ]:
categorical_columns = X_train.select_dtypes(include=["object", "category", "bool", "str"]).columns.tolist()
numeric_columns = X_train.select_dtypes(include=np.number).columns.tolist()

print("Categorical columns:", categorical_columns)
print("Numeric columns:", numeric_columns)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_columns),
        ("categorical", categorical_transformer, categorical_columns),
    ],
    remainder="drop",
)

### Question 4 — Pipeline reasoning

Why is the `ColumnTransformer` placed inside the model pipeline rather than fitted once to the complete dataset? What does `handle_unknown="ignore"` protect against?

**Answer:**  
TODO

## 5. Define baseline and classification models

We compare:

1. A **majority-class dummy model**.
2. Standard logistic regression.
3. Logistic regression with `class_weight="balanced"`.

Class weighting changes the fitting objective so errors on the minority class receive more influence. It does not create new observations and does not guarantee better calibrated probabilities.

In [ ]:
models = {
    "Dummy majority": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]),
    "Logistic regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "Weighted logistic regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Fitted: {name}")

## 6. Evaluate models on the validation set

For an imbalanced hazard problem, no single metric is sufficient.

- **Recall / sensitivity:** fraction of hazardous shifts detected.
- **Precision:** fraction of hazard alerts that are truly hazardous.
- **Specificity:** fraction of non-hazardous shifts correctly identified.
- **Balanced accuracy:** mean of sensitivity and specificity.
- **F1:** harmonic mean of precision and recall.
- **F2:** like F1, but gives recall more weight.
- **ROC-AUC:** ranking quality across thresholds; can appear optimistic under severe imbalance.
- **Average precision (PR-AUC summary):** summarizes the precision–recall curve and is often more informative for a rare positive class.

The function below evaluates probability-producing models at a specified threshold.

In [ ]:
def classification_metrics(y_true, y_probability, threshold=0.5):
    y_prediction = (np.asarray(y_probability) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_prediction, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "threshold": threshold,
        "precision": precision_score(y_true, y_prediction, zero_division=0),
        "recall": recall_score(y_true, y_prediction, zero_division=0),
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_prediction),
        "f1": f1_score(y_true, y_prediction, zero_division=0),
        "f2": fbeta_score(y_true, y_prediction, beta=2, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_probability),
        "average_precision": average_precision_score(y_true, y_probability),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
    }


def positive_probability(model, X_data):
    # Return P(y=1); fall back to hard predictions for non-probabilistic models.
    if hasattr(model, "predict_proba"):
        classes = model.classes_
        positive_index = int(np.where(classes == 1)[0][0])
        return model.predict_proba(X_data)[:, positive_index]
    return model.predict(X_data).astype(float)

In [ ]:
validation_rows = []
validation_probabilities = {}

for name, model in models.items():
    probabilities = positive_probability(model, X_val)
    validation_probabilities[name] = probabilities
    row = {"model": name, **classification_metrics(y_val, probabilities, threshold=0.5)}
    validation_rows.append(row)

validation_results = pd.DataFrame(validation_rows).set_index("model")
display(validation_results.round(3))

In [ ]:
# Confusion matrices at the default threshold of 0.5.
for name, probabilities in validation_probabilities.items():
    predictions = (probabilities >= 0.5).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_val,
        predictions,
        labels=[0, 1],
        display_labels=["No event", "High-energy event"],
        values_format="d",
    )
    plt.title(f"Validation confusion matrix: {name}")
    plt.tight_layout()
    plt.show()

### Question 5 — Validation comparison

Using the validation results:

1. Explain why the dummy model's ordinary accuracy could be high even if it detects no hazardous shifts. You may calculate accuracy separately, but do not use it as the primary selection metric.
2. Compare standard and weighted logistic regression at threshold 0.5.
3. Which model has better hazardous-class recall? What happened to precision and specificity?
4. Which metric would you prioritize for an early-warning screening system? Defend your choice while acknowledging the cost of false alarms.

**Answers:**  
TODO

## 7. Examine ROC and precision–recall curves

Curves show behavior across thresholds. The no-skill baseline for a precision–recall plot is the positive-class prevalence, not 0.5.

In [ ]:
# ROC curves for the two logistic-regression models.
for name in ["Logistic regression", "Weighted logistic regression"]:
    probabilities = validation_probabilities[name]
    fpr, tpr, _ = roc_curve(y_val, probabilities)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_val, probabilities):.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="No-skill")
plt.xlabel("False-positive rate")
plt.ylabel("True-positive rate / recall")
plt.title("Validation ROC curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Precision–recall curves.
for name in ["Logistic regression", "Weighted logistic regression"]:
    probabilities = validation_probabilities[name]
    precision, recall, _ = precision_recall_curve(y_val, probabilities)
    ap = average_precision_score(y_val, probabilities)
    plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})")

plt.axhline(y_val.mean(), linestyle="--", label=f"No-skill prevalence={y_val.mean():.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Validation precision–recall curves")
plt.legend()
plt.tight_layout()
plt.show()

### Question 6 — Curve interpretation

1. Why is the precision–recall baseline equal to prevalence?
2. Can two models have similar ROC-AUC but meaningfully different precision for the rare class? Explain.
3. Which curve is more directly useful for examining the trade-off between missed hazards and false alarms in this dataset?

**Answers:**  
TODO

## 8. Select a probability threshold using validation data

The default threshold of 0.5 is not automatically optimal. For this exercise, choose a threshold for the **weighted logistic regression** that maximizes F2 on the validation set. F2 emphasizes recall more than precision.

This is a model-selection step. Therefore, it must use validation data—not test data.

In [ ]:
selected_model_name = "Weighted logistic regression"
selected_model = models[selected_model_name]
val_probability = validation_probabilities[selected_model_name]

threshold_grid = np.linspace(0.01, 0.99, 99)
threshold_rows = [
    classification_metrics(y_val, val_probability, threshold=t)
    for t in threshold_grid
]
threshold_results = pd.DataFrame(threshold_rows)

best_index = threshold_results["f2"].idxmax()
best_threshold = float(threshold_results.loc[best_index, "threshold"])

print("Validation-selected threshold:", best_threshold)
display(
    threshold_results.loc[[best_index], [
        "threshold", "precision", "recall", "specificity",
        "balanced_accuracy", "f1", "f2", "false_positives", "false_negatives"
    ]].round(3)
)

In [ ]:
plt.plot(threshold_results["threshold"], threshold_results["precision"], label="Precision")
plt.plot(threshold_results["threshold"], threshold_results["recall"], label="Recall")
plt.plot(threshold_results["threshold"], threshold_results["f2"], label="F2")
plt.axvline(best_threshold, linestyle="--", label=f"Selected={best_threshold:.2f}")
plt.xlabel("Probability threshold")
plt.ylabel("Metric value")
plt.title("Validation metrics across thresholds")
plt.legend()
plt.tight_layout()
plt.show()

### Question 7 — Threshold choice

1. Compare the selected threshold with 0.5.
2. Explain how lowering or raising a threshold changes false positives and false negatives.
3. Is maximizing F2 necessarily the correct operational policy? Identify at least two pieces of domain information needed to define a defensible threshold.

**Answers:**  
TODO

## 9. Final evaluation on the untouched test set

At this point, the model family and threshold have been chosen. Evaluate them **once** on the test set. Do not return to the validation stage after seeing the test result.

For a strict introductory holdout workflow, we keep the fitted training-only model unchanged. In a production-oriented workflow, one might refit the fixed pipeline on combined train+validation data after all choices are locked, but then the threshold-selection procedure and final evaluation design must remain clearly documented.

In [ ]:
test_probability = positive_probability(selected_model, X_test)
test_metrics = classification_metrics(y_test, test_probability, threshold=best_threshold)

test_results = pd.DataFrame([test_metrics], index=[selected_model_name])
display(test_results.round(3))

final_test_prediction = (test_probability >= best_threshold).astype(int)
print(classification_report(
    y_test,
    final_test_prediction,
    target_names=["No event", "High-energy event"],
    digits=3,
    zero_division=0,
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    final_test_prediction,
    labels=[0, 1],
    display_labels=["No event", "High-energy event"],
    values_format="d",
)
plt.title("Final test confusion matrix")
plt.tight_layout()
plt.show()

### Question 8 — Final scientific interpretation

Write a concise results paragraph (approximately 150–250 words) that:

- identifies the final model and threshold-selection criterion;
- reports test recall, precision, specificity, balanced accuracy, F2, and average precision;
- states the numbers of false negatives and false positives;
- explains the practical meaning of those errors;
- avoids claiming that the model predicts rockbursts with certainty;
- states whether performance is adequate for deployment and why or why not.

**Results paragraph:**  
TODO

## 10. Reflection: imbalance remedies and study design

### Question 9 — Methods and limitations

Answer in approximately 200–300 words total.

1. Compare class weighting with random oversampling and random undersampling.
2. Where must resampling occur to avoid leakage when cross-validation is used?
3. Name two limitations of this dataset or workflow that restrict generalization to another mine, region, sensor network, or time period.
4. Propose one next modeling step and one next data-collection step.

**Answer:**  
TODO

## Grading rubric (25 points)

| Component | Points |
|---|---:|
| Reproducibility, loading, and data checks | 2 |
| Class-imbalance analysis | 3 |
| Correct train/validation/test split and leakage explanation | 2 |
| Preprocessing pipeline | 2 |
| Baseline and model fitting | 2 |
| Validation metrics and interpretation | 2 |
| Threshold selection using validation data | 2 |
| Final test evaluation | 2 |
| Scientific interpretation and limitations | 8 |

## 11. Reproducibility checklist

Before submitting, confirm that:

- [ ] The notebook runs from top to bottom without manual state from earlier sessions.
- [ ] All `TODO` sections are complete.
- [ ] Random seeds are fixed.
- [ ] Preprocessing is fitted only through pipelines on training data.
- [ ] Validation data, not test data, are used for model and threshold selection.
- [ ] The test set is evaluated only in the final section.
- [ ] Metrics are interpreted for the hazardous positive class.
- [ ] Figures have readable titles and axis labels.
- [ ] Scientific limitations and error costs are discussed.

## Dataset citation

Sikora, M., & Wrobel, L. (2010). Application of rule induction algorithms for analysis of data collected by seismic hazard monitoring systems in coal mines. *Archives of Mining Sciences, 55*(1), 91–114.